# Data Preparation — Washington DC Census Blocks & Socioeconomic Variables

Reconstructed from original analysis scripts (`bl_eco_adv.py`, `adv_acc_tweaked.py`, `weights_adv_acc_block.py`).

> **EDA lives in `03_eda_exploration.ipynb`** — this notebook only transforms and saves.

## Full pipeline from scratch

```
Step 1  Census Bureau REST API  →  ACS 5-year tract data (income, age groups)
Step 2  TIGER/Line             →  Census_Blocks_in_2020.shp  (block geometries + pop)
Step 3  gpd.sjoin(blocks, tracts, how="left", predicate="within")
                               →  blocks_with_income.shp     ← entry point below
Step 4  Disaggregate tract vars → block level via pop_weight
Step 5  Min-max normalise all variables [0, 1]
Step 6  Save                   →  blocksandtract_economic_final.shp
```

Steps 1–3 were completed in an earlier session (original scripts lost). The output `blocks_with_income.shp` is persisted in `data/shapefiles/`. Steps 4–6 are executed below.

## Sections
1. Imports & Paths
2. Original Data Build (Reference — not re-executed)
3. Load Entry Point — `blocks_with_income.shp`
4. Block-Level Disaggregation
5. Min-Max Normalisation
6. Save & Summary

In [3]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# Resolve paths relative to this notebook — works regardless of cwd
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_ROOT / "data" / "shapefiles"

print("Project root:", PROJECT_ROOT)
print("Data dir    :", DATA_DIR)

Project root: /Users/ushashi/Documents/codes/Accessibility_Research copy
Data dir    : /Users/ushashi/Documents/codes/Accessibility_Research copy/data/shapefiles


## 2. Original Data Build — Reference (Steps 1–3)

These steps were completed in the original research session. Code is provided for full reproducibility but **not re-executed** — `blocks_with_income.shp` already exists in `data/shapefiles/`.

---

### Step 1 — Census API Pull (ACS 5-year, 2020)

Tract-level variables pulled from the Census Bureau REST API for Washington DC (state FIPS = 11):

| ACS Code | Meaning | Shapefile column |
|---|---|---|
| `B19113_001E` | Median family income (tract) | `Median Fam` |
| `B01003_001E` | Total tract population | `Total Po_1` |
| `B01001_007E`–`B01001_049E` summed | Population aged 18–65 | `eighteento` |
| TIGER/Line (Census 2020 blocks) | Block total population | `Total Popu` |

```python
# import requests
# CENSUS_API_KEY = "YOUR_KEY"   # https://api.census.gov/data/key_signup.html
# BASE = "https://api.census.gov/data/2020/acs/acs5"
# age_cols = [f"B01001_{str(i).zfill(3)}E" for i in list(range(7,25)) + list(range(31,49))]
# variables = "B19113_001E,B01003_001E," + ",".join(age_cols)
# r = requests.get(BASE, params={"get": variables, "for": "tract:*", "in": "state:11", "key": CENSUS_API_KEY})
# tracts_df = pd.DataFrame(r.json()[1:], columns=r.json()[0])
# tracts_df["eighteento"] = tracts_df[age_cols].astype(float).sum(axis=1)
# tracts_df = tracts_df.rename(columns={"B19113_001E": "Median Fam", "B01003_001E": "Total Po_1"})
```

### Step 2 — Load Census Block Geometries (TIGER/Line)

```python
# blocks = gpd.read_file("Census_Blocks_in_2020.shp").to_crs("epsg:26985")
# Relevant columns: GEOID, Total Popu (block-level population count), geometry (polygon)
```

### Step 3 — Spatial Join: Blocks × Tracts → `blocks_with_income.shp`

Census blocks are always fully contained within their parent tract. `predicate="within"` is therefore exact — no block straddles tract boundaries.

```python
# tracts_gdf = gpd.GeoDataFrame(tracts_df, geometry=tracts_shp.geometry, crs="epsg:26985")
# blocks_with_income = gpd.sjoin(blocks, tracts_gdf, how="left", predicate="within")
# blocks_with_income.to_file(DATA_DIR / "blocks_with_income.shp")
```

---

## 3. Load Entry Point — `blocks_with_income.shp`

Output of Step 3 above. Loaded here as the starting point for disaggregation.

In [ ]:
df = gpd.read_file(DATA_DIR / "blocks_with_income.shp").to_crs("epsg:26985")
print(f"Shape  : {df.shape}")
print(f"CRS    : {df.crs}")
print(f"Columns: {list(df.columns)}\n")
df[["Total Popu", "Total Po_1", "Median Fam", "eighteento"]].describe()

Shape  : (6012, 63)
CRS    : EPSG:26985
Columns: ['OBJECTID', 'BLKGRP', 'BLOCK', 'GEOID_left', 'Total Popu', 'TRACT', 'ALAND_left', 'AWATER_lef', 'SHAPE_LENG', 'STUSAB', 'SUMLEV', 'GEOCODE_le', 'STATE', 'NAME_left', 'POP100_lef', 'HU100_left', 'SHAPEAREA_', 'SHAPELEN_l', 'index_righ', 'OBJECTID_x', 'TRACT_x', 'GEOID_righ', 'ALAND_righ', 'AWATER_rig', 'STUSAB_x', 'SUMLEV_x', 'GEOCODE_ri', 'STATE_x', 'NAME_right', 'POP100_rig', 'HU100_righ', 'SHAPEARE_1', 'SHAPELEN_r', 'Unnamed_ 0', 'OBJECTID_y', 'TRACT_y', 'ALAND_x', 'AWATER_x', 'STUSAB_y', 'SUMLEV_y', 'STATE_y', 'NAME_x', 'Longitude', 'Latitude', 'Field', 'Median Fam', 'Total Po_1', 'White', 'Black', 'American I', 'Asian', 'Native Haw', 'Some other', 'Total Po_2', 'Total Po_3', 'Total po_4', 'eighteento', '65 years a', 'MedianFami', 'TotalPopul', 'Accessibil', 'income_sha', 'geometry']



,OBJECTID,BLKGRP,BLOCK,GEOID_left,Total Popu,TRACT,ALAND_left,AWATER_lef,SHAPE_LENG,STUSAB,SUMLEV,GEOCODE_le,STATE,NAME_left,POP100_lef,HU100_left,SHAPEAREA_,SHAPELEN_l,index_righ,OBJECTID_x,TRACT_x,GEOID_righ,ALAND_righ,AWATER_rig,STUSAB_x,SUMLEV_x,GEOCODE_ri,STATE_x,NAME_right,POP100_rig,HU100_righ,SHAPEARE_1,SHAPELEN_r,Unnamed_ 0,OBJECTID_y,TRACT_y,ALAND_x,AWATER_x,STUSAB_y,SUMLEV_y,STATE_y,NAME_x,Longitude,Latitude,Field,Median Fam,Total Po_1,White,Black,American I,Asian,Native Haw,Some other,Total Po_2,Total Po_3,Total po_4,eighteento,65 years a,MedianFami,TotalPopul,Accessibil,income_sha,geometry
0,1,1,1004,7500000US110010001021004,0,000102,125930,0,1857.054502,DC,750,110010001021004,11,Block 1004,0,0,0,0,6,7,000102,11001000102,1706484,516665,DC,140,11001000102,11,Census Tract 1.02,3417,2053,0,0,6,7,102,1706484,516665,DC,140,11,Census Tract 1.02,-77.061272,38.905555,0,226773.0,3318,2868,34,0,150,0,147,1665,1653,329,2132,857,1.165477,-0.077246,100997.225543,0.000000,"POLYGON ((394049.812 138798.76, 394109.244 138..."
1,2,1,1006,7500000US110010001021006,265,000102,162560,0,2139.237670,DC,750,110010001021006,11,Block 1006,265,201,0,0,6,7,000102,11001000102,1706484,516665,DC,140,11001000102,11,Census Tract 1.02,3417,2053,0,0,6,7,102,1706484,516665,DC,140,11,Census Tract 1.02,-77.061272,38.905555,0,226773.0,3318,2868,34,0,150,0,147,1665,1653,329,2132,857,1.165477,-0.077246,100997.225543,18111.767631,"POLYGON ((394813.175 138308.122, 394906 138617..."
2,3,1,1001,7500000US110010001021001,0,000102,217073,0,2967.214792,DC,750,110010001021001,11,Block 1001,0,0,0,0,6,7,000102,11001000102,1706484,516665,DC,140,11001000102,11,Census Tract 1.02,3417,2053,0,0,6,7,102,1706484,516665,DC,140,11,Census Tract 1.02,-77.061272,38.905555,0,226773.0,3318,2868,34,0,150,0,147,1665,1653,329,2132,857,1.165477,-0.077246,100997.225543,0.000000,"POLYGON ((394162.164 138977.622, 394167.541 13..."
3,4,3,3009,7500000US110010014023009,163,001402,16256,0,643.263268,DC,750,110010014023009,11,Block 3009,163,131,0,0,32,33,001402,11001001402,895209,0,DC,140,11001001402,11,Census Tract 14.02,3514,1791,0,0,32,33,1402,895209,0,DC,140,11,Census Tract 14.02,-77.063096,38.960613,0,250001.0,3476,2564,416,0,111,35,215,1656,1820,846,1749,881,1.471180,0.058907,100997.221654,11723.292002,"POLYGON ((393845.418 143274.238, 393854.095 14..."
4,5,1,1009,7500000US110010024001009,60,002400,4697,0,388.957952,DC,750,110010024001009,11,Block 1009,60,21,0,0,43,44,002400,11001002400,556408,0,DC,140,11001002400,11,Census Tract 24,4095,1724,0,0,43,44,2400,556408,0,DC,140,11,Census Tract 24,-77.022408,38.941790,0,104278.0,4061,1852,1397,0,90,0,122,1996,2065,448,3127,486,-0.446679,0.563018,100997.213979,1540.674711,"POLYGON ((398143.683 141091.068, 398309.365 14..."


## 4. Block-Level Disaggregation

Tract variables are **constant within a tract** — every block in the same tract gets the same income value. We disaggregate to true block-level using population weight:

$$w_b = \frac{P_b^{\text{block}}}{P_t^{\text{tract}}} \qquad X_b = X_t \times w_b$$

High-population blocks carry more weight than low-population blocks in the same tract.

In [10]:
# Rename block pop column to match downstream script expectations
df["Bl_totalpo"] = df["Total Popu"].astype(float)

# Zero block populations → replace with median to avoid division by zero
median_bp = df["Bl_totalpo"].replace(0, np.nan).median()
df["Bl_totalpo"] = df["Bl_totalpo"].replace(0, median_bp)
print(f"Zero-pop blocks imputed with median: {median_bp:.0f}")

# Population weight of each block within its tract
df["pop_weight"] = df["Bl_totalpo"] / df["Total Po_1"].replace(0, np.nan)
df["pop_weight"] = df["pop_weight"].fillna(0)

# Disaggregate tract variables → block level
df["PerCapitaIncome_block"] = df["Median Fam"] * df["pop_weight"]   # → PerCapitaI in .shp
df["age_18to65_block"]      = df["eighteento"] * df["pop_weight"]   # → age_18to65 in .shp

print(f"\nPop-weight range : {df['pop_weight'].min():.4f} – {df['pop_weight'].max():.4f}")
df[["Bl_totalpo", "pop_weight", "PerCapitaIncome_block", "age_18to65_block"]].describe()

Zero-pop blocks imputed with median: 90

Pop-weight range : 0.0002 – 6.9231


,Bl_totalpo,pop_weight,PerCapitaIncome_block,age_18to65_block
count,6012.000000,6012.000000,5664.000000,6012.000000
mean,137.329508,0.258223,5851.061591,97.496011
std,181.256876,1.198627,9968.782172,140.729336
min,1.000000,0.000157,21.778772,0.689273
25%,62.000000,0.015957,1508.403483,38.642538
50%,90.000000,0.026205,3158.292808,61.067315
75%,134.000000,0.048283,6151.166365,92.133405
max,3060.000000,6.923077,182514.588817,2990.664653


## 5. Min-Max Normalisation

Scale all variables to [0, 1] so no single variable dominates the composite accessibility weight.

In [11]:
def minmax(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    return pd.Series(0.0, index=series.index) if hi == lo else (series - lo) / (hi - lo)

norm_map = {
    "Bl_totalpo"           : "norm_total_pop",
    "PerCapitaIncome_block": "norm_income",
    "age_18to65_block"     : "norm_age_18to65",
}
for raw, norm in norm_map.items():
    df[norm] = minmax(df[raw].fillna(0))

print("Normalised columns — range check (should all be 0.0–1.0):")
df[[*norm_map.values()]].agg(["min", "max"])

Normalised columns — range check (should all be 0.0–1.0):


,norm_total_pop,norm_income,norm_age_18to65
min,0.0,0.0,0.0
max,1.0,1.0,1.0


## 6. Save & Summary

In [12]:
OUTPUT_PATH = DATA_DIR / "blocksandtract_economic_final.shp"

keep_cols = ["geometry", "Bl_totalpo", "pop_weight",
             "PerCapitaIncome_block", "age_18to65_block",
             "norm_total_pop", "norm_income", "norm_age_18to65"]
keep_cols = [c for c in keep_cols if c in df.columns]

output_gdf = df[keep_cols].copy()
output_gdf.to_file(OUTPUT_PATH, encoding="UTF-8")
print(f"Saved {len(output_gdf):,} rows → {OUTPUT_PATH}")
print(f"Columns: {list(output_gdf.drop(columns='geometry').columns)}")

Saved 6,012 rows → /Users/ushashi/Documents/codes/Accessibility_Research copy/data/shapefiles/blocksandtract_economic_final.shp
Columns: ['Bl_totalpo', 'pop_weight', 'PerCapitaIncome_block', 'age_18to65_block', 'norm_total_pop', 'norm_income', 'norm_age_18to65']


In [13]:
print(f"Total blocks              : {len(output_gdf):,}")
print(f"Blocks with population > 0: {(output_gdf['Bl_totalpo'] > 0).sum():,}")
print(f"CRS                       : {output_gdf.crs}")
print()
output_gdf.drop(columns="geometry").describe().round(3)

Total blocks              : 6,012
Blocks with population > 0: 6,012
CRS                       : EPSG:26985



,Bl_totalpo,pop_weight,PerCapitaIncome_block,age_18to65_block,norm_total_pop,norm_income,norm_age_18to65
count,6012.000,6012.000,5664.000,6012.000,6012.000,6012.000,6012.000
mean,137.330,0.258,5851.062,97.496,0.045,0.030,0.032
std,181.257,1.199,9968.782,140.729,0.059,0.054,0.047
min,1.000,0.000,21.779,0.689,0.000,0.000,0.000
25%,62.000,0.016,1508.403,38.643,0.020,0.007,0.013
50%,90.000,0.026,3158.293,61.067,0.029,0.016,0.020
75%,134.000,0.048,6151.166,92.133,0.043,0.032,0.031
max,3060.000,6.923,182514.589,2990.665,1.000,1.000,1.000
